# LPGNet — released-design reconstruction and selected leakage conditions

The clean notebook selects **full** for manuscript reproduction: all four explicitly labelled conditions, ten outer folds, and the original epoch ceilings. `efficient` runs `local__inner` only and is not a full factorial; `factorial_fast` retains all four conditions at reduced epoch ceilings. The four conditions cross global/local control-reference LP fitting with test-driven/inner-only training control. Test-driven or globally fitted conditions intentionally demonstrate leakage and are **not** clean test estimates.

## Read before running: profile and budget
- Choose the profile after the original configuration definitions. The scientific settings are frozen before training; changing profile, cohort or search requires a **new checkpoint RUN_ID and fresh kernel**, never relabelling an existing `run_result`.
- `efficient`: ten folds, one declared training seed, three-candidate setting (unused by this fixed-architecture factorial runner), epoch caps **100/40**, stopping/LR patience **12/5**, and frozen-encoder caching. Only `local__inner`: 20 training stages, at most 1,400 epochs.
- `full`: original **1,000/200** epoch caps, patience **25/14**, no frozen cache, all four conditions. The original budget is **4 × 10 × 2 = 80 stages**, up to **48,000 epochs**. A small model is not a small experiment.
- `factorial_fast`: all four conditions, caps 100/40, patience 12/5, frozen cache: 80 stages, up to 5,600 epochs. `fixed` sets one configuration; this runner already has a fixed architecture (no configuration CV).
- The accelerated path caches **inference-mode frozen encoders**, disabling frozen stochastic noise/dropout compared with the full original protocol. It does **not** preserve the original training distribution. Modern Keras regularizers also preclude a claim of bit-identical historical reproduction.

## Verified off-runtime recovery
Run the storage cell immediately before the runner **even with RUN_EXPERIMENT=False**. Colab support mounts Drive through its API; non-Colab use fails clearly until an authenticated **rclone off-runtime remote** is configured (required on Kaggle). Never paste credentials into cells. Temporary working folders are not durable storage.

A verified upload/download probe and automatic remote restore precede training. Every completed epoch commits model/optimizer, history and stopping/LR state to verified off-runtime storage; condition/fold tasks also commit. Re-running storage reuses the matching mirror and scratch directory. Local scratch may disappear safely after a verified commit; a crash mid-epoch can lose at most the current uncommitted epoch. Cloud failure stops training. Provider availability and permanent retention are not guaranteed. Use only one writer per RUN_ID.

`RUN_EXPERIMENT=False` initially; `MAX_NEW_TASKS=1` limits each invocation. Repeat the runner to continue. Manual ZIP export is optional portability, not recovery safety; leave `RESTORE_FROM_ZIP=''` because the durable runner restores automatically.

Default `COHORT_POLICY='official_all'` retains 306 official recordings including known cross-ID duplicates; even `local__inner` is not identity-clean under that policy. Choose `curated` with a new RUN_ID for the curated 161-participant cohort. Inputs and clinical metadata are checksum verified; local LP uses only early-stopping-core controls.

**Source pin:** SOURCE_REV is `58e8d59dd6d1c9de5ef33164ac7f01f558b8aaf5`; its pinned support files are byte-identical to the archived experiment sources. Strict checks include durable storage; do not bypass pin failures. No training or new results are supplied by this notebook update.

In [ ]:
# Run this setup first in a fresh Kaggle/Colab kernel.
if 'run_result' in globals() or 'FROZEN_SCIENTIFIC_PROFILE' in globals():
    raise RuntimeError('Setup would reset a frozen run. Continue with the runner; use a fresh kernel and new RUN_ID for changes.')
import os, sys, tempfile, subprocess
from pathlib import Path
from collections import Counter
os.environ.setdefault('TF_FORCE_GPU_ALLOW_GROWTH','true')
SOURCE_REV = '58e8d59dd6d1c9de5ef33164ac7f01f558b8aaf5'  # Pinned committed support sources.
REPO_URL = 'https://github.com/AlexM3020/On-the-Validity-of-Machine-Learning-Evaluation-for-Gait--Based-Parkinson-s-Disease-Detection.git'
INSTALL_DEPENDENCIES = False
roots=[Path.cwd(),*Path.cwd().parents]
REPO_ROOT=next((p for b in roots for p in (b,b/'paper_repository') if (p/'gait/tf_notebook_support.py').is_file()),None)
if REPO_ROOT is None:
    REPO_ROOT=Path(tempfile.gettempdir())/('gait-keras-'+SOURCE_REV[:12])
    if not REPO_ROOT.exists(): subprocess.run(['git','clone','--no-checkout',REPO_URL,str(REPO_ROOT)],check=True)
    subprocess.run(['git','-C',str(REPO_ROOT),'checkout','--detach',SOURCE_REV],check=True)
REPO_ROOT=REPO_ROOT.resolve()
for filename in ('__init__.py','official_data.py','tf_notebook_support.py','durable_storage.py'):
    expected=subprocess.run(['git','-C',str(REPO_ROOT),'show',SOURCE_REV+':gait/'+filename],capture_output=True,check=True).stdout
    assert (REPO_ROOT/'gait'/filename).read_bytes().replace(b'\r\n',b'\n')==expected.replace(b'\r\n',b'\n'), 'Pinned source changed: '+filename
stale=[name for name,module in list(sys.modules.items()) if (name=='gait' or name.startswith('gait.')) and getattr(module,'__file__',None) and not Path(module.__file__).resolve().is_relative_to(REPO_ROOT)]
if stale: raise RuntimeError('Older gait imports cached: restart kernel and run this setup first. '+str(stale))
sys.path.insert(0,str(REPO_ROOT))
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable,'-m','pip','install','-r',str(REPO_ROOT/'requirements-tensorflow.txt')],check=True)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
keras=tf.keras
from scipy.signal import lfilter
from sklearn.metrics import (roc_auc_score,accuracy_score,balanced_accuracy_score,f1_score,confusion_matrix,ConfusionMatrixDisplay,log_loss,roc_curve)
from IPython.display import display
from gait import tf_notebook_support as nb
FAMILY='factorial'
SEED,FOLD_SEED,SPLIT_SEED,SEARCH_SEED=1,0,9137,20261117
N_FOLDS,N_INNER,VAL_FRAC=10,3,.15
LP_ORDER,WIN,STRIDE=11,100,50
BATCH_WIN,BATCH_REC=128,64
LR_WIN,LR_REC,LABEL_SMOOTH,L2_REG=5e-4,1e-3,.1,1e-3
MAX_EPOCHS_WIN,MAX_EPOCHS_REC=1000,200
PATIENCE,RLR_PATIENCE,MIN_LR=25,14,1e-7
N_BOOT=1000
BOOTSTRAP_SEED=2028
PAIRED_BOOTSTRAP_SEED=2028
THRESHOLD=.5
WORK_ROOT=Path('/kaggle/working') if Path('/kaggle').exists() else Path('/content') if Path('/content').exists() else Path.cwd()
# CHECKPOINT_DIR and OUTDIR are assigned by the verified storage cell, not to an ephemeral run directory here.
CACHE_DIR=Path(tempfile.gettempdir())/'gaitpdb-1.0.0-verified-cache'
COHORT_POLICY='official_all'  # Keeps original duplicates for reconstruction; not an identity-clean cohort.
ALLOW_DOWNLOAD=True
RUN_EXPERIMENT=False
MAX_NEW_TASKS=1
VERBOSE_EPOCHS=1
RESTORE_FROM_ZIP=''  # Legacy only: do not use with automatic durable restore.
EXPERIMENT_COMPLETE=False
summary=participant_ci=participant_fold_metrics=oof=fold_metrics=pd.DataFrame()
participants=participant_errors=pd_severity=by_study=seed_stability=pd.DataFrame()
keras.utils.set_random_seed(SEED)
print('Pinned source:',SOURCE_REV,'| TensorFlow:',tf.__version__,'| Keras:',keras.__version__)
print('GPU:',tf.config.list_physical_devices('GPU'))
print('Select the scientific profile, then verify off-runtime storage before enabling training.')
print('Four-condition profiles include deliberate leakage; efficient runs local__inner only, not a full factorial.')

In [ ]:
records,clinical,data_audit,data_provenance = nb.load_inputs(CACHE_DIR,COHORT_POLICY,ALLOW_DOWNLOAD)
subjects=sorted({r['subject'] for r in records})
subj_label={r['subject']:r['label'] for r in records}
by_file={r['file']:i for i,r in enumerate(records)}
by_subject={s:[i for i,r in enumerate(records) if r['subject']==s] for s in subjects}
print('Cohort policy:',COHORT_POLICY)
print(len(records),'recordings |',len(subjects),'participants |',dict(Counter(subj_label.values())))
print('Data provenance:',data_provenance)
print('Clinical unknowns are retained; no severity is inferred from filenames.')

In [ ]:
def preprocess(x):
    return lfilter([.5,.5],1,x,axis=0)[::2]

def fit_lp(control_sigs,order=LP_ORDER):
    return nb.fit_lp(control_sigs,order)

def apply_lp(x,C,stds):
    return nb.apply_lp(x,C,stds)

def lp_features(indices,C,stds):
    return {i:apply_lp(records[i]['proc'],C,stds) for i in indices}

for r in records:
    r['proc']=preprocess(r['signal'])
print('Fixed label-free downsampling prepared. LP fitting is deferred to the declared experimental condition.')

In [ ]:
def make_folds():
    return nb.outer_folds(records,N_FOLDS,FOLD_SEED)

def split_val(indices,frac=VAL_FRAC,seed=SEED):
    return nb.subject_split(records,indices,frac,seed)

def labels(indices):
    return np.array([records[i]['label'] for i in indices],np.float32)

FOLDS=make_folds()
for k,f in enumerate(FOLDS):
    print('Fold',k,'train participants:',len({records[i]['subject'] for i in f['train']}),
          'test participants:',len({records[i]['subject'] for i in f['test']}))

In [ ]:
# Supported Keras API; models and training are version-pinned in the shared helper.
def build_lpgnet(**kwargs):
    return nb.build_lpgnet(**kwargs)

def build_recording_model(window_model):
    return nb.build_recording_model(window_model)

MODEL_READY=False
smoke=nb.smoke_check(FAMILY,{})
print('Real forward/backward and model/optimizer reload:',smoke)
MODEL_READY=True
print('Training uses bounded-memory batches and automatic atomic epoch checkpoints after RUN_EXPERIMENT=True.')
print('No obsolete workers/use_multiprocessing arguments are passed to Keras fit.')

In [ ]:
# ============================================================
# OOF metrics, participant aggregation, errors, and clinical subgroups
# ============================================================
def _validated_binary_inputs(y, p):
    y = np.asarray(y)
    p = np.asarray(p)
    if y.ndim != 1 or p.ndim != 1 or len(y) == 0 or len(y) != len(p):
        raise ValueError('Labels and probabilities must be nonempty 1-D arrays of equal length.')
    if np.iscomplexobj(y) or not np.isin(y, [0, 1]).all():
        raise ValueError('Labels must contain only binary values 0 and 1.')
    if np.iscomplexobj(p):
        raise ValueError('Probabilities must be real numbers in [0, 1].')
    p = np.asarray(p, float)
    if not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():
        raise ValueError('Probabilities must be finite and in [0, 1].')
    return y.astype(int), p

def metric_dict(y, p):
    y, p = _validated_binary_inputs(y, p)
    yh = (p >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yh, labels=[0,1]).ravel()
    return dict(n=len(y), accuracy=accuracy_score(y, yh),
                balanced_accuracy=balanced_accuracy_score(y, yh),
                macro_f1=f1_score(y, yh, labels=[0,1], average='macro', zero_division=0),
                f1_pd=f1_score(y, yh, zero_division=0),
                sensitivity=tp/(tp+fn) if tp+fn else np.nan,
                specificity=tn/(tn+fp) if tn+fp else np.nan,
                roc_auc=roc_auc_score(y, p) if len(np.unique(y)) == 2 else np.nan,
                log_loss=log_loss(y, np.clip(p,1e-7,1-1e-7), labels=[0,1]),
                brier=float(np.mean((p-y)**2)))

def oof_frame(rows):
    df = pd.DataFrame(rows)
    y, p = _validated_binary_inputs(df['label'], df['probability'])
    df['label'] = y; df['probability'] = p
    df['prediction'] = (df['probability'] >= 0.5).astype(int)
    df['correct'] = (df['prediction'] == df['label']).astype(int)
    df['confidence'] = np.maximum(df['probability'], 1-df['probability'])
    df['nll'] = -(df['label']*np.log(np.clip(df['probability'],1e-7,1)) +
                  (1-df['label'])*np.log(np.clip(1-df['probability'],1e-7,1)))
    # Strict metadata lookup: missing subjects or fields raise KeyError.
    for col in ('study','diagnosis','hy','hy_label','hy_status','age','gender'):
        df[col] = df['subject'].map(lambda s: clinical[s][col])
    return df

def participant_oof(df, group_cols):
    """Equal weight per participant; probability is averaged over that person's walks."""
    keys = group_cols + ['subject']
    df = df.copy()
    y, p = _validated_binary_inputs(df['label'], df['probability'])
    df['label'] = y; df['probability'] = p
    if df[keys + ['fold']].isna().any().any():
        raise ValueError('Participant grouping keys and folds must not be missing.')
    grouped = df.groupby(keys, as_index=False)
    counts = grouped[['label', 'fold']].nunique(dropna=False)
    if (counts['label'] > 1).any():
        raise ValueError('Conflicting labels for a subject within an evaluation group.')
    if (counts['fold'] > 1).any():
        raise ValueError('Multiple folds for a subject within an evaluation group.')
    if 'file' in df.columns and df.duplicated(subset=group_cols + ['file']).any():
        raise ValueError('Duplicate file within an evaluation group.')
    agg = dict(label=('label','first'), fold=('fold','first'), probability=('probability','mean'),
               recordings=('file','nunique') if 'file' in df.columns else ('subject','size'))
    out = grouped.agg(**agg)
    return oof_frame(out.to_dict('records'))

def summarize_oof(df, group_cols):
    rows = []
    for keys, g in df.groupby(group_cols):
        if not isinstance(keys, tuple): keys = (keys,)
        base = dict(zip(group_cols, keys))
        rec = metric_dict(g['label'], g['probability'])
        part = participant_oof(g, group_cols)
        pm = metric_dict(part['label'], part['probability'])
        rows.append({**base, **{'recording_'+k:v for k,v in rec.items()}})
        rows[-1].update({'participant_'+k:v for k,v in pm.items()})
    return pd.DataFrame(rows)

def wilson_interval(correct, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    ph = correct/n; den = 1+z*z/n
    center = (ph+z*z/(2*n))/den
    half = z*np.sqrt(ph*(1-ph)/n+z*z/(4*n*n))/den
    return center-half, center+half

def error_tables(df, group_cols):
    part = participant_oof(df, group_cols)
    errors = part[part['correct'] == 0].sort_values(group_cols+['nll'], ascending=[True]*len(group_cols)+[False])
    pd_rows = []
    for keys, g in part[part['label'] == 1].groupby(group_cols+['hy_label'], dropna=False):
        if not isinstance(keys, tuple): keys = (keys,)
        base = dict(zip(group_cols+['hy_label'], keys))
        n = len(g); correct = int(g['correct'].sum()); lo, hi = wilson_interval(correct, n)
        pd_rows.append({**base, 'pd_participants':n, 'correct':correct, 'errors':n-correct,
                        'pd_sensitivity':correct/n, 'wilson_95_low':lo, 'wilson_95_high':hi})
    study_rows = []
    for keys, g in part.groupby(group_cols+['study','diagnosis']):
        if not isinstance(keys, tuple): keys = (keys,)
        base = dict(zip(group_cols+['study','diagnosis'], keys))
        study_rows.append({**base, **metric_dict(g['label'], g['probability'])})
    return part, errors, pd.DataFrame(pd_rows), pd.DataFrame(study_rows)

def show_table(df, title, limit=100):
    print('\n'+title)
    display(df.head(limit))

def pooled_bootstrap(df, group_cols, metrics=('accuracy','balanced_accuracy','macro_f1','sensitivity','specificity','roc_auc')):
    """Stratified participant bootstrap conditional on fitted OOF predictions."""
    rows = []
    for keys, g in df.groupby(group_cols):
        if not isinstance(keys, tuple): keys = (keys,)
        base = dict(zip(group_cols, keys))
        part = participant_oof(g, group_cols).reset_index(drop=True)
        y = part['label'].to_numpy(); p = part['probability'].to_numpy()
        class_idx = [np.flatnonzero(y == c) for c in (0,1)]
        rng = np.random.default_rng(BOOTSTRAP_SEED)
        for metric in metrics:
            observed = metric_dict(y,p)[metric]
            vals = []
            for _ in range(N_BOOT):
                ii = np.concatenate([rng.choice(ix, len(ix), replace=True) for ix in class_idx])
                vals.append(metric_dict(y[ii],p[ii])[metric])
            rows.append({**base,'metric':metric,'estimate':observed,
                         'bootstrap_95_low':np.nanpercentile(vals,2.5),
                         'bootstrap_95_high':np.nanpercentile(vals,97.5)})
    return pd.DataFrame(rows)

def plot_confusion_and_roc(df, group_cols, title_col):
    part = participant_oof(df, group_cols)
    groups = list(part.groupby(group_cols))
    fig, axes = plt.subplots(len(groups), 2, figsize=(11,4.3*len(groups)), squeeze=False)
    for row, (keys,g) in enumerate(groups):
        if not isinstance(keys, tuple): keys=(keys,)
        title = ' | '.join(f'{c}={v}' for c,v in zip(group_cols,keys))
        cm = confusion_matrix(g['label'],g['prediction'],labels=[0,1])
        ConfusionMatrixDisplay(cm,display_labels=['Control','PD']).plot(ax=axes[row,0],colorbar=False)
        axes[row,0].set_title(title+' | participant OOF')
        fpr,tpr,_ = roc_curve(g['label'],g['probability'])
        axes[row,1].plot(fpr,tpr,label=f"AUC={roc_auc_score(g['label'],g['probability']):.3f}")
        axes[row,1].plot([0,1],[0,1],'--',color='gray')
        axes[row,1].set(xlabel='False-positive rate',ylabel='True-positive rate',title=title)
        axes[row,1].legend()
    fig.tight_layout(); plt.show()


In [ ]:
# Declare the scientific budget AFTER original configuration definitions, BEFORE storage/training.
RUN_PROFILE='full'  # Manuscript protocol. Options: efficient, full, factorial_fast, fixed. Changes need a new RUN_ID.
REQUIRE_GPU=True
_PROFILE_KEYS=('SOURCE_REV','FAMILY','SEED','FOLD_SEED','SPLIT_SEED','SEARCH_SEED','N_FOLDS','N_INNER',
    'VAL_FRAC','LP_ORDER','WIN','STRIDE','BATCH_WIN','BATCH_REC','PRED_BATCH','LR_WIN','LR_REC',
    'LABEL_SMOOTH','L2_REG','MAX_EPOCHS_WIN','MAX_EPOCHS_REC','MAX_EPOCHS_MIL','PATIENCE',
    'RLR_PATIENCE','MIN_LR','TRAIN_SEEDS','N_TRIALS','THRESHOLD','COHORT_POLICY','SPACE_A',
    'SPACE_RESMIL','EXPORT_ATTENTION','RUN_PROFILE','CACHE_FROZEN_ENCODER','CONDITIONS',
    'N_BOOT','BOOTSTRAP_SEED','PAIRED_BOOTSTRAP_SEED')
def scientific_profile(namespace):
    return nb.json_ready({k:namespace[k] for k in _PROFILE_KEYS if k in namespace})
_proposed=scientific_profile({**globals(),**nb.profile_settings(FAMILY,RUN_PROFILE)})
if 'run_result' in globals() and 'FROZEN_SCIENTIFIC_PROFILE' not in globals():
    raise RuntimeError('Existing run_result has no frozen profile. Use a fresh kernel and new checkpoint RUN_ID; do not relabel results.')
if 'FROZEN_SCIENTIFIC_PROFILE' in globals() and _proposed!=globals()['FROZEN_SCIENTIFIC_PROFILE']:
    raise RuntimeError('Scientific profile changed. Start a fresh kernel with a new checkpoint RUN_ID.')
globals().update(nb.profile_settings(FAMILY,RUN_PROFILE))
CONDITIONS=globals()['CONDITIONS']  # Explicit binding for notebook static analysis after namespace update.
FROZEN_SCIENTIFIC_PROFILE=scientific_profile(globals())
print('Frozen profile:',RUN_PROFILE,'| conditions:',CONDITIONS)
print('Budget:',nb.budget_report(globals(),FAMILY))
print('Original four-condition budget: 4 x 10 x 2 = 80 training stages; 48,000 maximum epochs. Small model != small experiment.')
print('Efficient is local__inner only, NOT a full factorial; full and factorial_fast retain all four conditions.')
print('Cached frozen encoders run in inference mode: frozen noise/dropout is disabled, unlike the full original protocol; training distributions are not identical.')

In [ ]:
# REQUIRED preflight, even when RUN_EXPERIMENT=False. This cell never starts training.
RUN_ID='factorial_efficient_v3'  # Unique per scientific plan; one runtime/writer per ID.
STORAGE_BACKEND='colab_drive'
DRIVE_ROOT='/content/drive/MyDrive/gait-checkpoints'
RCLONE_REMOTE=''  # Kaggle: choose 'rclone' and a real authenticated remote, e.g. researchdrive:gait-checkpoints.
# Configure rclone authentication outside notebook cells; never paste credentials or tokens here.
REQUIRE_DURABLE_STORAGE=True
if RESTORE_FROM_ZIP:
    raise RuntimeError('Leave RESTORE_FROM_ZIP empty. Durable storage restores automatically; do not mix manual ZIP state.')
_storage_settings=(STORAGE_BACKEND,DRIVE_ROOT,RCLONE_REMOTE)
if globals().get('DURABLE_RUN_ID')!=RUN_ID or globals().get('DURABLE_MIRROR') is None:
    if 'run_result' in globals():
        raise RuntimeError('Existing run_result cannot be rebound to another mirror. Restart the kernel; use the same RUN_ID to resume or a new ID for a new plan.')
    CHECKPOINT_DIR=Path(tempfile.mkdtemp(prefix='gait-scratch-'))
    DURABLE_MIRROR=nb.connect_storage(CHECKPOINT_DIR,RUN_ID,backend=STORAGE_BACKEND,
        drive_root=DRIVE_ROOT,rclone_remote=RCLONE_REMOTE)
    DURABLE_RUN_ID=RUN_ID
    DURABLE_STORAGE_SETTINGS=_storage_settings
else:
    DURABLE_MIRROR=globals()['DURABLE_MIRROR']
    CHECKPOINT_DIR=Path(globals()['CHECKPOINT_DIR'])
    if globals().get('DURABLE_STORAGE_SETTINGS')!=_storage_settings:
        raise RuntimeError('Storage settings changed for this RUN_ID. Restart and reconnect explicitly; do not mix backends.')
    if DURABLE_MIRROR.local_root!=CHECKPOINT_DIR.resolve():
        raise RuntimeError('Mirror does not match CHECKPOINT_DIR. Restart and restore into fresh scratch.')
    DURABLE_MIRROR.verified=False  # A failed re-probe must not leave a stale success flag.
    DURABLE_MIRROR.verify()  # Repeat upload/download probe, without overwriting local progress.
if not DURABLE_MIRROR.verified:
    raise RuntimeError('Storage verification failed; training is disabled.')
OUTDIR=CHECKPOINT_DIR/'reports'
print('Verified durable run:',DURABLE_RUN_ID,'| local scratch:',CHECKPOINT_DIR)
print('Automatic restore complete. Epoch commits are verified off-runtime; cloud failures stop training. No training started.')

In [ ]:
# Execute only the selected conditions; local__inner excludes validation controls from LP fitting.
if RUN_EXPERIMENT:
    assert MODEL_READY,'Run the real model checkpoint smoke test first'
    if scientific_profile(globals())!=FROZEN_SCIENTIFIC_PROFILE:
        raise RuntimeError('Frozen scientific profile changed; use a fresh kernel and new RUN_ID.')
    if RESTORE_FROM_ZIP:
        raise RuntimeError('Do not use manual ZIP restore; the storage cell restores committed state automatically.')
    assert REQUIRE_DURABLE_STORAGE and REQUIRE_GPU,'Keep storage and GPU requirements enabled'
    assert DURABLE_RUN_ID==RUN_ID and DURABLE_STORAGE_SETTINGS==(STORAGE_BACKEND,DRIVE_ROOT,RCLONE_REMOTE),'Rerun storage preflight'
    assert DURABLE_MIRROR.verified and DURABLE_MIRROR.local_root==Path(CHECKPOINT_DIR).resolve(),'Verify the matching storage mirror first'
    experiment=nb.NotebookExperiment(globals(),FAMILY)  # Engine consumes DURABLE_MIRROR and verifies requirements.
    run_result=experiment.factorial(max_tasks=MAX_NEW_TASKS)
    globals().update(nb.unpack_result(globals(),run_result,FAMILY))
    print('Complete condition/fold tasks:',len(run_result['fold_rows']),'/',len(CONDITIONS)*N_FOLDS)
    print('Experiment complete:',EXPERIMENT_COMPLETE)
else:
    EXPERIMENT_COMPLETE=False
    print('Training disabled. Verify storage even now; set RUN_EXPERIMENT=True only after profile and preflight checks.')

In [ ]:
if EXPERIMENT_COMPLETE:
    globals().update(nb.summary_tables(globals(),FAMILY))
    show_table(summary.round(4),'Complete pooled recording and participant OOF metrics for selected conditions')
    show_table(participant_ci.round(4),'Participant-stratified bootstrap intervals, conditional on fitted predictions')
    show_table(participant_fold_metrics.round(4),'Participant fold metrics; fold SD is not a confidence interval')
    if 'global__test' in set(fold_metrics['condition']):
        paper=pd.DataFrame({'paper_reported_mean':[.917,.903,.932],'paper_reported_sd':[.094,.058,.036]},index=['roc_auc','accuracy','f1_pd'])
        current=fold_metrics[fold_metrics.condition=='global__test'][['roc_auc','accuracy','f1_pd']].agg(['mean','std']).T
        paper['reconstructed_mean']=current['mean']; paper['reconstructed_sd']=current['std']
        show_table(paper.round(4),'Published versus modernized global/test condition; report profile and cache protocol, not exact reproduction')
    else:
        print('No published global/test comparison: global__test was not run in this profile.')
else:
    print('Complete every selected condition/fold before interpreting results; efficient is not a full factorial.')

In [ ]:
if EXPERIMENT_COMPLETE:
    globals().update(nb.error_reports(globals(),FAMILY))
    paired_comparisons=nb.paired_tables(globals(),FAMILY)  # Engine uses only conditions actually selected.
    if paired_comparisons.empty:
        print('Only one selected condition: no paired condition contrast is available.')
    else:
        show_table(paired_comparisons.round(4),'Paired selected-condition differences, first minus second')
        print('Positive accuracy/AUC favors the first condition; negative log loss/Brier favors the first.')
        print('Leakage contrasts are not independent clinical-validation claims.')
else:
    print('Paired comparisons require completion of every selected condition/fold.')

In [ ]:
if EXPERIMENT_COMPLETE:
    globals().update(nb.error_reports(globals(),FAMILY))
    plot_confusion_and_roc(oof,['condition'],'condition')
    show_table(participants.sort_values('nll',ascending=False).round(4),'All participant OOF errors and successes',limit=200)
    show_table(participant_errors.round(4),'Misclassified participants',limit=200)
    show_table(pd_severity.round(4),'PD sensitivity by verified HY stage (unknown retained)')
    show_table(by_study.round(4),'Study-by-diagnosis metrics')
    show_table(seed_stability.round(4),'Errors recurring across factorial conditions (not training seeds)')
else:
    print('Complete all conditions before reviewing outer errors.')

In [ ]:
# Optional portability/report exports; verified remote checkpoints already provide automatic recovery.
EXPORT_CHECKPOINT_BUNDLE=False
EXPORT_ANALYSIS_CSV=False
if EXPORT_CHECKPOINT_BUNDLE:
    print('Optional checkpoint ZIP:',nb.export_checkpoint_bundle(CHECKPOINT_DIR))
    print('Download this optional copy if wanted; do not mix manual ZIP restore with the durable runner.')
if EXPORT_ANALYSIS_CSV:
    if not EXPERIMENT_COMPLETE: raise RuntimeError('Analysis export requires complete OOF results')
    nb.export_tables(globals(),FAMILY)
if not EXPORT_CHECKPOINT_BUNDLE and not EXPORT_ANALYSIS_CSV:
    print('No extra export requested. Training epoch/task state is committed automatically to verified off-runtime storage.')

## Reporting and persistence limits

- Report frozen profile, selected condition labels, all folds and cohort policy with every result. `efficient` is local/inner only, not a full factorial. `global` LP uses held-out controls; `test` validation uses held-out outcomes for stopping. These are deliberate leakage conditions, not deployable evaluation pipelines.
- `official_all` retains cross-ID duplicates: even local/inner is not certified identity-clean under this policy.
- Keras 3 regularizers, bounded-memory batches and inference-mode frozen caching are explicit protocol choices. Topology/two-stage structure alone do not imply historical numerical reproduction or identical stochastic training distributions.
- Pair only available conditions by participant on identical folds. One condition provides no pairwise contrast. Published global/test comparisons are shown only when that condition is present. Bootstrap intervals condition on fitted predictions; fold SD is not repeated-training uncertainty.
- Checkpoints bind data/source/configuration. Do not edit CSVs to mark completion, merge runs, change profiles in place or reuse another cohort's RUN_ID.
- Per-epoch verified off-runtime commits make local scratch disposable after commit. Automatic restore replaces manual ZIP recovery. A mid-epoch crash can lose the current epoch; storage failure stops training. Cloud-provider availability and permanent retention are not guaranteed.
- Smoke checks validate actual models and serialization on the chosen runtime. This update does not supply new full-cohort training results.